In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import snapatac2 as snap

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
samples = list(np.loadtxt(f'{projdir}/txt/samples.txt', dtype=str))
s = samples[0]

In [ ]:
meta = pd.read_csv(f'{projdir}/csv/atac/archr/ambimux_atac_qc.csv', sep='\t', header=0, index_col=0)
meta.shape

In [ ]:
meta.head()

In [ ]:
meta['archr_barcode'] = meta.index
meta['sample'] = [x.split('#')[0] for x in meta.index]
meta['barcode'] = [x.split('#')[1] for x in meta.index]
meta.index = [f'{x}_{y}' for x,y in zip(meta['barcode'], meta['sample'])]

In [ ]:
meta.head()

### ambimux

In [ ]:
ambimux = pd.read_csv(f'{projdir}/csv/demux/ambimux/wgs/ambimux_joint_sings.csv', sep='\t', header=0, index_col=0)
ambimux.shape

In [ ]:
m_mask = meta.index.isin(ambimux.index)
a_mask = ambimux.index.isin(meta.index)

In [ ]:
np.sum(m_mask), np.sum(a_mask)

In [ ]:
tmp_ambimux = ambimux[a_mask]
tmp_meta = meta[m_mask]
tmp_ambimux = tmp_ambimux.reindex(tmp_meta.index)

In [ ]:
tmp_meta.shape, tmp_ambimux.shape

In [ ]:
np.sum(tmp_meta.index.isin(tmp_ambimux.index))

In [ ]:
tmp_meta['assignment'] = ambimux.reindex(tmp_meta.index)['assignment']
tmp_meta['ambient_rna'] = ambimux.reindex(tmp_meta.index)['best_rna_ambient']
tmp_meta['ambient_dna'] = ambimux.reindex(tmp_meta.index)['best_atac_ambient']

In [ ]:
tmp_meta.head()

### scrublet
- need to have run http://localhost:6430/notebooks/project-cluo/igvf/2023_YR2/multiome/ipynb/atac/snapatac2/01a_setup_snapatac2_h5ad.ipynb

In [ ]:
tmp_meta.shape

In [ ]:
scrublet = pd.read_csv(f'{projdir}/csv/doublets/atac/wgs/merged.scrublet.csv', sep='\t', index_col=0, header=0)
scrublet.shape

In [ ]:
tmp_scrublet = scrublet.reindex(tmp_meta.index)
np.sum(tmp_scrublet.index == tmp_meta.index)

In [ ]:
tmp_scrublet.columns

In [ ]:
tmp_meta['doublet_score'] = tmp_scrublet.reindex(tmp_meta.index)['doublet_score']
tmp_meta['doublet_probability'] = tmp_scrublet.reindex(tmp_meta.index)['doublet_probability']

In [ ]:
tmp_meta.head()

In [ ]:
### probability > 0.5 is the default filter
snap.pp.filter_doublets

In [ ]:
tmp_meta.shape

In [ ]:
tmp_meta.columns

In [ ]:
tmp_scrublet

In [ ]:
### aim for 2.5% doublets because we already removed them with ambimux
percentile = 0.975
dbl_cutoff = tmp_scrublet['doublet_probability'].quantile(percentile)
mask = tmp_scrublet['doublet_probability'] > dbl_cutoff
tmp_scrublet['is_doublet'] = mask
sum(mask) / tmp_scrublet.shape[0]

In [ ]:
# tmp_meta['is_doublet'] = tmp_meta['doublet_probability'] > 0.5
tmp_meta['is_doublet'] = tmp_meta['doublet_probability'] > dbl_cutoff
np.sum(tmp_meta['is_doublet']) / tmp_meta.shape[0]

### plot QC

In [ ]:
select_cols = ['TSSEnrichment', 'ReadsInPromoter', 'nFrags', 'ambient_dna', 'doublet_probability']

tmp_cols = np.tile(select_cols, 10)
# tmp_samples = np.tile(samples, 4)
tmp_samples = np.repeat(samples, 5)

xlims = np.tile([15, 10000, 20000, 1.0, 1.0], 10)

fig, axes = plt.subplots(10, 5, figsize=(14, 20))

for i, ax in enumerate(axes.flatten()):
    col = tmp_cols[i]
    s = tmp_samples[i]
    tmp_plot = tmp_meta[tmp_meta['sample'] == s]
    sns.histplot(data=tmp_plot, x=col, ax=ax)
    xlim = xlims[i]
    ax.set_xlim(0, xlim)
    
    ax.grid(False)
    ax.set_xlabel(ax.get_xlabel(), fontsize=10)
    ax.set_ylabel('')
    ax.set_title(s)
    
plt.subplots_adjust(wspace=0.5, hspace=1.0)

In [ ]:
tmp_meta.head()

In [ ]:
demux_singlets = ambimux['sample'].value_counts().sort_index()
pd.DataFrame(demux_singlets)

In [ ]:
nfrags_filter = 1000
tss_filter = 5
ambient_dna_filter = 0.20
doublet_filter = 0.50

select_cols = ['nFrags', 'TSSEnrichment', 'doublet_probability', 'ambient_dna']

check_meta = pd.DataFrame(columns=select_cols, index=samples)
for s in samples:
    tmp = tmp_meta[tmp_meta['sample'] == s]
    add = [np.sum(tmp['nFrags'] < nfrags_filter), 
            np.sum(tmp['TSSEnrichment'] < tss_filter),
           np.sum(~tmp['is_doublet']),
        np.sum(tmp['ambient_dna'] > ambient_dna_filter)]
    check_meta.loc[s] = add
    
check_meta

In [ ]:
n_drops = tmp_meta['Sample'].value_counts().sort_index()
prop_meta = pd.DataFrame([check_meta.iloc[i,:] / n_drops[i] for i in range(len(samples))])
prop_meta

In [ ]:
tmp_meta['archr_barcode'] = [f'{x}#{y}' for x,y in zip(tmp_meta['Sample'], tmp_meta['barcode'])]

In [ ]:
# nfrags_filter = 1000
nfrags_filter = 500
tss_filter = 2
doublet_filter = 0.5
ambient_dna_filter = 0.50

meta = tmp_meta.copy()
all_mask = (meta['nFrags'] > nfrags_filter) & \
            (meta['TSSEnrichment'] > tss_filter) & \
           (~meta['is_doublet']) & \
        (meta['ambient_dna'] < ambient_dna_filter)
np.sum(all_mask)

In [ ]:
np.sum(meta['nFrags'] > nfrags_filter), \
np.sum(meta['TSSEnrichment'] > tss_filter), \
np.sum(~meta['is_doublet']), \
np.sum(meta['ambient_dna'] < ambient_dna_filter)

In [ ]:
filtered_meta = tmp_meta[all_mask]
filtered_meta.shape

In [ ]:
filtered_meta['archr_barcode'].head()

In [ ]:
filtered_meta.index.name = 'barcode_sample'

In [ ]:
filtered_meta.head()

In [ ]:
filtered_meta.shape

In [ ]:
tmp_meta.to_csv(f'{projdir}/csv/qc/atac/atac_qc_all.csv', sep='\t', header=True, index=True)
filtered_meta.to_csv(f'{projdir}/csv/qc/atac/atac_qc_filtered.csv', sep='\t', header=True, index=True)

In [ ]:
!date